# Post-training Qwen2.5-0.5B on GSM8K: SFT, on-policy DPO, GRPO (RLVR)

Code: https://github.com/Haryanfe777/qwen-0.5b-sft-dpo-grpo. Each section below is added in the build
step that uses it, after that step is verified.

**Notebook settings (right panel) before running:** Accelerator **GPU T4 x2** (the code uses one GPU),
Internet **on**, Persistence **Files only** (keeps `/kaggle/working` across restarts).
Optional: Add-ons > Secrets > `HF_TOKEN` (avoids Hugging Face rate limits; needed later to publish).

## Setup (run at the start of every session)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # one T4 is enough for a 0.5B model
os.environ["TOKENIZERS_PARALLELISM"] = "false"
REPO_URL = "https://github.com/Haryanfe777/qwen-0.5b-sft-dpo-grpo.git"
WORK = "/kaggle/working/qwen-0.5b-sft-dpo-grpo"
if os.path.exists(WORK):
    !git -C {WORK} pull -q --ff-only
else:
    !git clone -q {REPO_URL} {WORK}
%cd {WORK}
!git log --oneline -1

In [ ]:
# Pinned versions from pyproject.toml; torch stays Kaggle's build. The last lines show any pip conflicts.
!pip install -q -e ".[eval,dev]" 2>&1 | tail -15

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("No HF_TOKEN secret, continuing anonymously:", type(e).__name__)

## Step 3: environment check
Unit tests, then versions, GPU, the real data loader, and one greedy answer from the base model.
Must end with `ENV OK`.

In [ ]:
!python -m pytest -q 2>&1 | tail -3
!python scripts/check_env.py